# base

> The reset, the document on the tokens, and prose

In [ ]:
#| default_exp base

The base layer styles native HTML elements on the tokens, so a page written as plain semantic markup is readable before any component is involved. It has three parts: a reset that removes the browser's default styling, document styles for text, links, code, lists, and tables, and typrose for long-form prose. Each registers with `reg.rule` in the `reset` or `base` layer.

In [ ]:
from fastcore.utils import *
from importlib.resources import files
from answerui.registry import reg
import answerui.tokens  # installs Registry.context

In [ ]:
from fasthtml.common import *
from answerui.demo import demo

Everything in this notebook has a light and a dark rendering, so a helper shows both at once. Each panel is an island with its own `color-scheme`, and the page and text colours re-derive inside it:

In [ ]:
def schemes(*fts):
    "The same markup in a light and a dark island, side by side"
    panels = [Div(*fts, style=f'color-scheme:{s};background:var(--n-3);color:var(--n2);padding:var(--space)') for s in ('light', 'dark')]
    return Div(*panels, style='display:grid;grid-template-columns:repeat(2, minmax(0, 1fr))')

## Reset

The browser's default stylesheet puts margins on headings and paragraphs, gives form controls their own font, and lets images overflow. We undo these once, here. Margins go to zero on every block element, at zero specificity so any later rule wins. Spacing then comes only from a layout container or a prose region, so a parent controls the gaps between its children. Controls inherit the surrounding font and colour, and media is constrained to its container.

In [ ]:
reg.rule('reset', r"""
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
""", layer='reset', doc='Border-box sizing, no block margins, media constrained, controls inheriting text.')

reset [reset]: Border-box sizing, no block margins, media constrained, controls inheriting text.

## Document

The page takes the base rung and text the top of the neutral ladder, so the whole document follows `--hue` and `color-scheme`. Headings, links, and code read the type and colour tokens directly. `body` is a size container, so components can query the width they actually have rather than the viewport. `accent-color` and `caret-color` make native controls follow the hue before any control is styled.

Native scrollbars use the muted neutral for their thumb and a transparent track. They follow the surrounding palette without changing the browser's scrollbar width or behaviour.

A `mark` is a warning context, so its wash and ink come from the warning family.

In [ ]:
reg.context('mark', 'var(--hue-warning)')
reg.rule('document', r"""
:root { accent-color: var(--v); caret-color: var(--v); background: var(--n-3); }
* { scrollbar-color: var(--n1) transparent; }
body { color: var(--n2); font: var(--weight) var(--text)/var(--leading) var(--font); container-type: inline-size; }
h1, h2, h3, h4, h5, h6 { font-weight: var(--weight1); line-height: var(--leading-1); text-wrap: balance; }
h1 { font-size: var(--text3); } h2 { font-size: var(--text2); } h3 { font-size: var(--text1); }
p { text-wrap: pretty; }
a { color: var(--v); }
code, kbd, pre, samp { font-family: var(--font-mono); }
code, kbd { font-size: .875em; padding: .1em .3em; border-radius: var(--radius-1); }
code { background: var(--n-2); }
kbd { border: 1px solid var(--n-1); box-shadow: var(--raised); }
pre { background: var(--n-2); padding: var(--space); border-radius: var(--radius); overflow-x: auto; }
pre code { font-size: inherit; padding: 0; background: none; }
ul, ol { padding-inline-start: var(--space1); }
blockquote { border-inline-start: 3px solid var(--n-1); padding-inline-start: var(--space); color: var(--n1); }
hr { border: 0; border-block-start: 1px solid var(--n-1); }
small, figcaption { font-size: var(--text-1); }
figcaption { color: var(--n1); }
table { border-collapse: collapse; inline-size: 100%; }
th, td { padding: var(--space-2) var(--space-1); text-align: start; }
th { font-weight: var(--weight1); color: var(--n1); }
thead { border-block-end: 1px solid var(--n); }
tbody tr { border-block-end: 1px solid var(--n-1); }
:focus-visible { outline: 2px solid var(--v-1); outline-offset: 2px; }
mark { background: var(--v-3); color: var(--v2); padding: .1em .2em; border-radius: var(--radius-1); }
""", layer='base', doc='The page, text, headings, links, code, lists, quotes, tables, rules, and marks on the tokens.')

document [base]: The page, text, headings, links, code, lists, quotes, tables, rules, and marks on the tokens.

One more rule belongs to the page rather than to any component. A component sets its own `display`, and by layer order that beats the browser's rule for the `hidden` attribute, so a hidden card or alert would stay on screen. Native state has to win, so the attribute gets a rule in the utilities layer, above every component. The `until-found` form is left to the browser, which reveals such content when a find-in-page or fragment navigation lands in it:

In [ ]:
reg.rule('hidden', '[hidden]:not([hidden=until-found]) { display: none; }', layer='utilities', doc='The hidden attribute wins over any display a component sets.')

hidden [utilities]: The hidden attribute wins over any display a component sets.

A sample with nothing but content elements. None of them carries a margin, so the sample spaces its children with a grid gap, as a layout container would:

In [ ]:
sample = Div(
    H1('A document on the ladder'),
    P('Body text with ', A('a link', href='#'), ', ', Code('code'), ', ', Kbd('Ctrl'), ' and a ', Mark('marked'), ' phrase.'),
    H2('Lists, quotes, and tables'),
    Ul(Li('Bullets take the hairline rung'), Li('Captions and quotes take the muted rung')),
    Blockquote('Quoted text sits on a hairline and drops to the muted rung.'),
    Pre(Code('print("code blocks sit on the raised surface")')),
    Table(Thead(Tr(Th('Rung'), Th('Job'))), Tbody(Tr(Td('n2'), Td('text')), Tr(Td('v'), Td('links')))),
    Hr(), Small('Small print.'),
    style='display:grid;gap:var(--space)')
demo(schemes(sample))

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
[project] { --hue: 300; }
mark { --hue: var(--hue-warning); }
:user-invalid, [aria-invalid=true] { --hue: var(--hue-danger); }
:root, [danger], [warning], [success], [info], [project], mark, :user-invalid, [aria-invalid=true] {
--n-3: light-dark(oklch(0.9914 clamp(0, 0.00801 + 0.00551*cos((1*var(--hue) - 136.63)*1deg) + 0.00329*cos((2*var(--hue) - 268.97)*1deg) + 0.00139*cos((3*var(--hue) - 327.76)*1deg) + 0.00123*cos((4*var(--hue) - 93.61)*1deg) + 0.00125*cos((5*var(--hue) - 211.17)*1deg) + 0.00034*cos((6*var(--hue) - 339.8)*1deg) + 0.00023*cos((7*var(--hue) - 56.25)*1deg) + 0.00038*cos((8*var(--hue) - 125.04)*1deg) + 0.00015*cos((9*var(--hue) - 283.35)*1deg) + 7e-05*cos((10*var(--hue) - 349.22)*1deg) + 0.00021*cos((11*var(--hue) - 12.11)*1deg) + 0.00015*cos((12*var(--hue) - 161.94)*1deg), 0.02) var(--hue)), oklch(0.1026 clamp(0, 0.01959 + 0.00041*cos((1*var(--hue) - 19.99)*1deg), 0.02) var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9506 0.02 var(--hue)), oklch(0.1434 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7903 0.02 var(--hue)), oklch(0.3385 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6503 0.02 var(--hue)), oklch(0.4611 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5247 0.02 var(--hue)), oklch(0.5812 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4217 0.02 var(--hue)), oklch(0.693 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.913 0.03581 var(--hue)), oklch(0.1825 clamp(0, 0.05701 + 0.02563*cos((1*var(--hue) - 325.45)*1deg) + 0.01252*cos((2*var(--hue) - 263.8)*1deg) + 0.00819*cos((3*var(--hue) - 96.1)*1deg) + 0.00256*cos((4*var(--hue) - 85.33)*1deg) + 0.0017*cos((5*var(--hue) - 319.1)*1deg) + 0.0034*cos((6*var(--hue) - 157.26)*1deg) + 0.00073*cos((7*var(--hue) - 138.05)*1deg) + 0.00112*cos((8*var(--hue) - 2.43)*1deg) + 0.00177*cos((9*var(--hue) - 237.19)*1deg) + 0.00037*cos((10*var(--hue) - 228.33)*1deg) + 0.00064*cos((11*var(--hue) - 69.15)*1deg), 0.09221) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7453 clamp(0, 0.12952 + 0.00279*cos((1*var(--hue) - 18.52)*1deg), 0.13231) var(--hue)), oklch(0.3792 clamp(0, 0.11554 + 0.05156*cos((1*var(--hue) - 327.56)*1deg) + 0.02507*cos((2*var(--hue) - 264.93)*1deg) + 0.01578*cos((3*var(--hue) - 92.55)*1deg) + 0.00467*cos((4*var(--hue) - 86.48)*1deg) + 0.00337*cos((5*var(--hue) - 312.62)*1deg) + 0.00706*cos((6*var(--hue) - 156.72)*1deg) + 0.00165*cos((7*var(--hue) - 144.83)*1deg) + 0.00212*cos((8*var(--hue) - 5.61)*1deg) + 0.00327*cos((9*var(--hue) - 233.15)*1deg), 0.18274) var(--hue))); /* wash hover, selected */
--v-1: ligh

## Prose

Long-form content wants more than chrome: rhythm between blocks, a reading measure, and typographic care in lists, quotes, code, and tables. That is [typrose](https://github.com/AnswerDotAI/typrose)'s job. It is a CSS-native port of Tailwind's typography plugin, scoped under a `prose` class, with every colour read from a `--prose-*` property on that scope.

We bundle the file and rewrite its selectors at registration, since the framework marks scopes with attributes: `.prose` becomes `[prose]` and a size variant such as `.prose-lg` becomes `[prose="lg"]`, quoted because a value like `2xl` is not valid unquoted, so `Article(prose=True)` and `Article(prose='lg')` both match. Its grey theme classes and the dark-mode `invert` block go at the same time, because the ladder does both of those jobs:

In [ ]:
TYPROSE = files('answerui').joinpath('css/typrose.css').read_text()
TYPROSE = re.sub(r'(/\*[^*]*\*/\n)?\.prose-(slate|zinc|neutral|stone|invert) \{[^}]*\}\n*', '', TYPROSE)
TYPROSE = re.sub(r'\.prose-(\w+)', r'[prose="\1"]', TYPROSE)
TYPROSE = re.sub(r'\.prose\b', '[prose]', TYPROSE)
reg.rule('typrose', TYPROSE, layer='base', doc='Prose typography from typrose, on the `prose` attribute.')

typrose [base]: Prose typography from typrose, on the `prose` attribute.

typrose's own colours are a fixed grey scale. Re-declaring its fourteen properties on `[prose]`, later in the same layer, maps each onto a rung, so prose follows the hue, the scheme, and any context it sits in. Two are left alone: the `hr` divider icon is an inline SVG with a fixed stroke, and the `kbd` shadow is a fixed black alpha. The same rule caps a prose region at a reading measure, which typrose leaves to the page, and sets its font size from `--text` times typrose's own size setting, in place of the fixed `1rem` typrose starts from. Its rhythm stays typrose's: `--prose-scale` spaces prose relative to its type, `--space` spaces chrome and layout, and `--text` is the base size the two share:

In [ ]:
reg.rule('prose', r"""
[prose] {
  max-inline-size: 65ch; font-size: calc(var(--text) * var(--prose-font-scale));
  --prose-body: var(--n2); --prose-headings: var(--n2); --prose-bold: var(--n2); --prose-quotes: var(--n2); --prose-code: var(--n2);
  --prose-links: var(--v);
  --prose-lead: var(--n1); --prose-counters: var(--n1); --prose-captions: var(--n1);
  --prose-bullets: var(--n-1); --prose-hr: var(--n-1); --prose-quote-borders: var(--n-1); --prose-td-borders: var(--n-1);
  --prose-th-borders: var(--n);
}
""", layer='base', doc='typrose colours mapped onto the ladder, its size from `--text`, and the reading measure.')

prose [base]: typrose colours mapped onto the ladder, its size from `--text`, and the reading measure.

The same content as an article, with rhythm and measure from typrose and colour from the ladder:

In [ ]:
article = Article(*sample.children, prose=True)
demo(schemes(article))

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
[project] { --hue: 300; }
mark { --hue: var(--hue-warning); }
:user-invalid, [aria-invalid=true] { --hue: var(--hue-danger); }
:root, [danger], [warning], [success], [info], [project], mark, :user-invalid, [aria-invalid=true] {
--n-3: light-dark(oklch(0.9914 clamp(0, 0.00801 + 0.00551*cos((1*var(--hue) - 136.63)*1deg) + 0.00329*cos((2*var(--hue) - 268.97)*1deg) + 0.00139*cos((3*var(--hue) - 327.76)*1deg) + 0.00123*cos((4*var(--hue) - 93.61)*1deg) + 0.00125*cos((5*var(--hue) - 211.17)*1deg) + 0.00034*cos((6*var(--hue) - 339.8)*1deg) + 0.00023*cos((7*var(--hue) - 56.25)*1deg) + 0.00038*cos((8*var(--hue) - 125.04)*1deg) + 0.00015*cos((9*var(--hue) - 283.35)*1deg) + 7e-05*cos((10*var(--hue) - 349.22)*1deg) + 0.00021*cos((11*var(--hue) - 12.11)*1deg) + 0.00015*cos((12*var(--hue) - 161.94)*1deg), 0.02) var(--hue)), oklch(0.1026 clamp(0, 0.01959 + 0.00041*cos((1*var(--hue) - 19.99)*1deg), 0.02) var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9506 0.02 var(--hue)), oklch(0.1434 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7903 0.02 var(--hue)), oklch(0.3385 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6503 0.02 var(--hue)), oklch(0.4611 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5247 0.02 var(--hue)), oklch(0.5812 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4217 0.02 var(--hue)), oklch(0.693 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.913 0.03581 var(--hue)), oklch(0.1825 clamp(0, 0.05701 + 0.02563*cos((1*var(--hue) - 325.45)*1deg) + 0.01252*cos((2*var(--hue) - 263.8)*1deg) + 0.00819*cos((3*var(--hue) - 96.1)*1deg) + 0.00256*cos((4*var(--hue) - 85.33)*1deg) + 0.0017*cos((5*var(--hue) - 319.1)*1deg) + 0.0034*cos((6*var(--hue) - 157.26)*1deg) + 0.00073*cos((7*var(--hue) - 138.05)*1deg) + 0.00112*cos((8*var(--hue) - 2.43)*1deg) + 0.00177*cos((9*var(--hue) - 237.19)*1deg) + 0.00037*cos((10*var(--hue) - 228.33)*1deg) + 0.00064*cos((11*var(--hue) - 69.15)*1deg), 0.09221) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7453 clamp(0, 0.12952 + 0.00279*cos((1*var(--hue) - 18.52)*1deg), 0.13231) var(--hue)), oklch(0.3792 clamp(0, 0.11554 + 0.05156*cos((1*var(--hue) - 327.56)*1deg) + 0.02507*cos((2*var(--hue) - 264.93)*1deg) + 0.01578*cos((3*var(--hue) - 92.55)*1deg) + 0.00467*cos((4*var(--hue) - 86.48)*1deg) + 0.00337*cos((5*var(--hue) - 312.62)*1deg) + 0.00706*cos((6*var(--hue) - 156.72)*1deg) + 0.00165*cos((7*var(--hue) - 144.83)*1deg) + 0.00212*cos((8*var(--hue) - 5.61)*1deg) + 0.00327*cos((9*var(--hue) - 233.15)*1deg), 0.18274) var(--hue))); /* wash hover, selected */
--v-1: ligh

Inside a `danger` context the whole article re-derives, because every colour it reads is a rung:

In [ ]:
demo(schemes(Div(article, danger=True)))

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
[project] { --hue: 300; }
mark { --hue: var(--hue-warning); }
:user-invalid, [aria-invalid=true] { --hue: var(--hue-danger); }
:root, [danger], [warning], [success], [info], [project], mark, :user-invalid, [aria-invalid=true] {
--n-3: light-dark(oklch(0.9914 clamp(0, 0.00801 + 0.00551*cos((1*var(--hue) - 136.63)*1deg) + 0.00329*cos((2*var(--hue) - 268.97)*1deg) + 0.00139*cos((3*var(--hue) - 327.76)*1deg) + 0.00123*cos((4*var(--hue) - 93.61)*1deg) + 0.00125*cos((5*var(--hue) - 211.17)*1deg) + 0.00034*cos((6*var(--hue) - 339.8)*1deg) + 0.00023*cos((7*var(--hue) - 56.25)*1deg) + 0.00038*cos((8*var(--hue) - 125.04)*1deg) + 0.00015*cos((9*var(--hue) - 283.35)*1deg) + 7e-05*cos((10*var(--hue) - 349.22)*1deg) + 0.00021*cos((11*var(--hue) - 12.11)*1deg) + 0.00015*cos((12*var(--hue) - 161.94)*1deg), 0.02) var(--hue)), oklch(0.1026 clamp(0, 0.01959 + 0.00041*cos((1*var(--hue) - 19.99)*1deg), 0.02) var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9506 0.02 var(--hue)), oklch(0.1434 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7903 0.02 var(--hue)), oklch(0.3385 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6503 0.02 var(--hue)), oklch(0.4611 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5247 0.02 var(--hue)), oklch(0.5812 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4217 0.02 var(--hue)), oklch(0.693 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.913 0.03581 var(--hue)), oklch(0.1825 clamp(0, 0.05701 + 0.02563*cos((1*var(--hue) - 325.45)*1deg) + 0.01252*cos((2*var(--hue) - 263.8)*1deg) + 0.00819*cos((3*var(--hue) - 96.1)*1deg) + 0.00256*cos((4*var(--hue) - 85.33)*1deg) + 0.0017*cos((5*var(--hue) - 319.1)*1deg) + 0.0034*cos((6*var(--hue) - 157.26)*1deg) + 0.00073*cos((7*var(--hue) - 138.05)*1deg) + 0.00112*cos((8*var(--hue) - 2.43)*1deg) + 0.00177*cos((9*var(--hue) - 237.19)*1deg) + 0.00037*cos((10*var(--hue) - 228.33)*1deg) + 0.00064*cos((11*var(--hue) - 69.15)*1deg), 0.09221) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7453 clamp(0, 0.12952 + 0.00279*cos((1*var(--hue) - 18.52)*1deg), 0.13231) var(--hue)), oklch(0.3792 clamp(0, 0.11554 + 0.05156*cos((1*var(--hue) - 327.56)*1deg) + 0.02507*cos((2*var(--hue) - 264.93)*1deg) + 0.01578*cos((3*var(--hue) - 92.55)*1deg) + 0.00467*cos((4*var(--hue) - 86.48)*1deg) + 0.00337*cos((5*var(--hue) - 312.62)*1deg) + 0.00706*cos((6*var(--hue) - 156.72)*1deg) + 0.00165*cos((7*var(--hue) - 144.83)*1deg) + 0.00212*cos((8*var(--hue) - 5.61)*1deg) + 0.00327*cos((9*var(--hue) - 233.15)*1deg), 0.18274) var(--hue))); /* wash hover, selected */
--v-1: ligh

The consumers agree. A compact region sets the two base tokens and reads `--text` for its own size, and its heading, control, and prose follow, with prose keeping its own rhythm:

In [ ]:
def region(title, style=''):
    "A heading, a paragraph, a native control, and a prose region"
    return Div(H2(title), P('Body text at the region\'s size.'), Input(placeholder='A native control'),
        Article(P('Prose sets its size from the same base and keeps its own rhythm.'), P('A second paragraph shows that rhythm.'), prose=True),
        style=f'font-size:var(--text);display:grid;gap:var(--space);padding:var(--space);{style}')

demo(Div(region('Normal'), region('Compact', style='--space:.75rem;--text:.875rem'), style='display:grid;grid-template-columns:1fr 1fr'))

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
[project] { --hue: 300; }
mark { --hue: var(--hue-warning); }
:user-invalid, [aria-invalid=true] { --hue: var(--hue-danger); }
:root, [danger], [warning], [success], [info], [project], mark, :user-invalid, [aria-invalid=true] {
--n-3: light-dark(oklch(0.9914 clamp(0, 0.00801 + 0.00551*cos((1*var(--hue) - 136.63)*1deg) + 0.00329*cos((2*var(--hue) - 268.97)*1deg) + 0.00139*cos((3*var(--hue) - 327.76)*1deg) + 0.00123*cos((4*var(--hue) - 93.61)*1deg) + 0.00125*cos((5*var(--hue) - 211.17)*1deg) + 0.00034*cos((6*var(--hue) - 339.8)*1deg) + 0.00023*cos((7*var(--hue) - 56.25)*1deg) + 0.00038*cos((8*var(--hue) - 125.04)*1deg) + 0.00015*cos((9*var(--hue) - 283.35)*1deg) + 7e-05*cos((10*var(--hue) - 349.22)*1deg) + 0.00021*cos((11*var(--hue) - 12.11)*1deg) + 0.00015*cos((12*var(--hue) - 161.94)*1deg), 0.02) var(--hue)), oklch(0.1026 clamp(0, 0.01959 + 0.00041*cos((1*var(--hue) - 19.99)*1deg), 0.02) var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9506 0.02 var(--hue)), oklch(0.1434 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7903 0.02 var(--hue)), oklch(0.3385 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6503 0.02 var(--hue)), oklch(0.4611 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5247 0.02 var(--hue)), oklch(0.5812 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4217 0.02 var(--hue)), oklch(0.693 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.913 0.03581 var(--hue)), oklch(0.1825 clamp(0, 0.05701 + 0.02563*cos((1*var(--hue) - 325.45)*1deg) + 0.01252*cos((2*var(--hue) - 263.8)*1deg) + 0.00819*cos((3*var(--hue) - 96.1)*1deg) + 0.00256*cos((4*var(--hue) - 85.33)*1deg) + 0.0017*cos((5*var(--hue) - 319.1)*1deg) + 0.0034*cos((6*var(--hue) - 157.26)*1deg) + 0.00073*cos((7*var(--hue) - 138.05)*1deg) + 0.00112*cos((8*var(--hue) - 2.43)*1deg) + 0.00177*cos((9*var(--hue) - 237.19)*1deg) + 0.00037*cos((10*var(--hue) - 228.33)*1deg) + 0.00064*cos((11*var(--hue) - 69.15)*1deg), 0.09221) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7453 clamp(0, 0.12952 + 0.00279*cos((1*var(--hue) - 18.52)*1deg), 0.13231) var(--hue)), oklch(0.3792 clamp(0, 0.11554 + 0.05156*cos((1*var(--hue) - 327.56)*1deg) + 0.02507*cos((2*var(--hue) - 264.93)*1deg) + 0.01578*cos((3*var(--hue) - 92.55)*1deg) + 0.00467*cos((4*var(--hue) - 86.48)*1deg) + 0.00337*cos((5*var(--hue) - 312.62)*1deg) + 0.00706*cos((6*var(--hue) - 156.72)*1deg) + 0.00165*cos((7*var(--hue) - 144.83)*1deg) + 0.00212*cos((8*var(--hue) - 5.61)*1deg) + 0.00327*cos((9*var(--hue) - 233.15)*1deg), 0.18274) var(--hue))); /* wash hover, selected */
--v-1: ligh

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()